> 在线讲义：[第 6 章 · 实时调度：动作块、异步推理、快慢两套系统](https://wikieden.github.io/robot-edge-inference/lessons/06-realtime-scheduling.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 6 章 · 实时调度：动作块、异步推理、快慢两套系统

配套讲义：仓库里 `ai-infra-edge/notes/lessons/06-realtime-scheduling.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（73 ms、25 步、50 Hz、g = 0.7 来自论文；300 ms 和时延预算各项是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 同步停着的比例 | $73 / (73 + 25 \times 20)$ | `sync_idle_fraction(73, 25, 20)` | 12.7% |
| 第一步 推理变慢 | $300 / 800$ | `sync_idle_fraction(300, 25, 20)` | 37.5% |
| 第二步 异步预算 | $0.7 \times 50 \times 20$ | `async_budget_ms(50, 0.7, 20)` | 700 ms |
| 第四步 时延预算 | 各段相加 | `budget_total(parts)` | 114.3 ms |

π0 的 73 ms、25 步、50 Hz 和 SmolVLA 的 g = 0.7 来自论文；300 ms 和时延预算表里的各项是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `sync_idle_fraction`**：同步执行：做完 exec_steps 步动作后停下等推理，停着的时间占几成。

In [ ]:
def sync_idle_fraction(infer_ms: float, exec_steps: int, dt_ms: float) -> float:
    """同步执行：做完 exec_steps 步动作后停下等推理，停着的时间占几成。"""
    return infer_ms / (infer_ms + exec_steps * dt_ms)

**函数 `async_budget_ms`**：异步执行：队列还剩 g × chunk 步时就发起下一次推理，推理必须在这些步用完前回来。

In [ ]:
def async_budget_ms(chunk: int, g: float, dt_ms: float) -> float:
    """异步执行：队列还剩 g × chunk 步时就发起下一次推理，推理必须在这些步用完前回来。"""
    return g * chunk * dt_ms

**函数 `budget_total`**：时延预算：从拍照到动作发出，各段加起来。

In [ ]:
def budget_total(parts: dict[str, float]) -> float:
    """时延预算：从拍照到动作发出，各段加起来。"""
    return sum(parts.values())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：同步执行时机器人停着等的时间（π0：做 25 步、每步 20 ms）")
for infer in (73, 300):
    f = sync_idle_fraction(infer, 25, 20)
    print(f"  推理 {infer:>3} ms → 每轮 {infer + 500} ms 里停 {infer} ms，占 {f:.1%}")
print("  （300 ms 是编的，代表放到小板子上变慢后的情况）")

In [ ]:
print("\n第二步：异步执行的时间预算（动作块 50 步、每步 20 ms）")
for g in (0.0, 0.5, 0.7):
    print(f"  g = {g}：队列剩 {g * 50:.0f} 步时触发 → 推理必须在 {async_budget_ms(50, g, 20):.0f} ms 内回来")

In [ ]:
print("\n第四步：一条时延预算（各项是编的，自己的机器要实测 p99）")
parts = {"等下一帧相机（30 fps 最坏）": 1000 / 30, "预处理": 5, "模型推理": 73, "后处理": 2, "发给控制器": 1}
for k, v in parts.items():
    print(f"  {k:<18} {v:6.1f} ms")
print(f"  合计 {budget_total(parts):.1f} ms")

## 3. 代码里才看得到的

- `async_budget_ms` 假设推理时间每次一样。真实推理有抖动，要拿 p99 的推理时间和预算比，不能拿平均值。
- 动画里的队列模拟把新块直接接在旧队列后面（每次补 25 步），是简化；SmolVLA 的实现会在重叠部分合并新旧动作。

## 4. 自己改一改

- 把 `g` 改成 0.3，看预算变成多少，π0 在 4090 上的 73 ms 还够不够。
- 在 `parts` 里加一项"日志进程抢 CPU 10 ms"，看合计变成多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 同步做完停下等，推理越慢停越多；
> 异步剩 g 就发起，L 小于 g H 乘 Δt；
> 新旧两块要接平，重叠平均或 RTC；
> 慢的想来快的动，预算看 p99。